In [0]:
jsonPath = "/Volumes/main/default/bronze"
schemaLocation = "/Volumes/main/default/schema"
schemaHint = "id INT, name STRING, age INT"

# Nota: El checkpoint no se define en la lectura, se guarda para el writeStream posterior 
checkPoint = "/Volumes/main/default/checkpoint" 

dfRead=(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    
    # 1. Esquema y Hints [cite: 105, 247]
    .option("cloudFiles.schemaLocation", schemaLocation)
    .option("cloudFiles.schemaHints", schemaHint)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # [cite: 10, 240, 244]
    
    # 2. Control de Flujo (Backpressure)
    .option("cloudFiles.maxFilesPerTrigger", 10) # Cambiado a 10 para mejor paralelismo (1 es muy lento)
    .option("cloudFiles.maxBytesPerTrigger", "10m") # Soporta sintaxis humana como "10m" (10 Megas) o "1g"
    
    # 3. Descubrimiento de Archivos
    .option("cloudFiles.useNotifications", "false") # 'false' usa Directory Listing (recomendado para empezar sin configurar IAM)
    .option("cloudFiles.includeExistingFiles", "true") # Lee los históricos que ya estén en la carpeta la primera vez
    
    # 4. Mecanismo de redundancia (Solo si useNotifications fuera "true")
    # .option("cloudFiles.backfillInterval", "1 day") # Valor óptimo de producción por defecto
    
    .load(jsonPath)
)

In [0]:
## Transformations
# Filtramos para asegurar que el ID no sea nulo y la edad sea un rango lógico
df_limpio = dfRead.filter("id IS NOT NULL AND age >= 0 AND age < 120")

In [0]:
## Reading Declarative (PySpark Structured Streaming Standard)
(df_limpio.writeStream
 .format("delta")
 .option("checkpointLocation", checkPoint) # Estándar oficial de Databricks
 .trigger(availableNow=True) # Procesa incrementalmente y se apaga (Ahorro de DBU) /  ProcessinTime = "X SECONDS" / continuous='1 SECOND"
 .outputMode("append") # 'complete' para agregaciones, 'append' para paso directo. Tendría que ser complete en el caso de que se agregara un campo nuevo
 .toTable("main.default.table_destino")
)